# Lab type: debug
# Course: ML302 — Transformer Models & Fine-Tuning
# Lesson: Tokenisation and Positional Encoding
# Task: The code below contains 2 bugs. Both run without raising a Python error, but produce silently wrong results. Find each bug, explain it in the markdown cell below the buggy code, and fix it.

In [ ]:
# Install dependencies (uncomment if running on Colab)
# !pip install transformers torch

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

print('Setup complete.')

## Bug 1: Tokeniser/model mismatch

A team member is building a BERT-based sentiment classifier. They load the model correctly, then use what they call a 'simpler tokeniser' for speed. The code runs, outputs logits, and no error is raised. But the predictions are meaningless.

Run the cell and observe the output.

In [ ]:
# Bug 1: tokeniser and model are from different families
model = AutoModelForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2,
)

# Using GPT-2 tokeniser for 'speed'
tokenizer = AutoTokenizer.from_pretrained('gpt2')

text = 'The product quality is excellent.'
inputs = tokenizer(text, return_tensors='pt')

print('Token IDs:', inputs['input_ids'])
print('Keys in inputs:', list(inputs.keys()))

with torch.no_grad():
    # May raise KeyError or produce garbage logits
    try:
        outputs = model(**inputs)
        print('Logits:', outputs.logits)
        print('NOTE: these logits are meaningless — the input representation is corrupted')
    except Exception as e:
        print('Error:', e)

**Explain Bug 1:** Why does using the GPT-2 tokeniser with a BERT model corrupt the input representation, even though no error is raised? In your answer, explain what token IDs represent and what happens when the IDs don't match the model's embedding table.

*(Write your answer here.)*

In [ ]:
# Fix Bug 1: always load tokeniser from the same checkpoint as the model
model_name = 'bert-base-uncased'
tokenizer_fixed = AutoTokenizer.from_pretrained(model_name)
model_fixed = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

text = 'The product quality is excellent.'
inputs_fixed = tokenizer_fixed(text, return_tensors='pt')

print('Token IDs:', inputs_fixed['input_ids'])
print('Decoded: ', tokenizer_fixed.decode(inputs_fixed['input_ids'][0]))
print('Keys in inputs:', list(inputs_fixed.keys()))

with torch.no_grad():
    outputs_fixed = model_fixed(**inputs_fixed)
print('Logits (random init head, but correct representation):', outputs_fixed.logits)

## Bug 2: Right-truncation drops label signal

A team member builds a review classification pipeline. The reviews are long and must be truncated to 128 tokens. The model is trained and achieves decent training loss. At evaluation, accuracy on the test set is surprisingly low — much lower than expected given how quickly training loss decreased.

Run the cell to see which tokens survive truncation.

In [ ]:
# Bug 2: default right-truncation removes the verdict at the end of long reviews
tokenizer_bert = AutoTokenizer.from_pretrained('bert-base-uncased')

# A product review where all sentiment signal is at the end
review = (
    'I bought this laptop three months ago and have been using it daily for work. '
    'The build quality seems reasonable for the price point. '
    'Battery life is acceptable for short meetings. '
    'The keyboard is standard and takes some getting used to. '
    'Performance on typical office tasks is adequate. '
    'The display brightness could be better in direct sunlight. '
    'I have noticed some fan noise during heavier workloads. '
    'Software setup was straightforward with no major issues. '
    'After using it extensively, my final verdict: '
    'Absolutely terrible thermal management — throttles to 40% CPU after 10 minutes.'
)

# Default truncation: right-truncate (drops from the end)
inputs_default = tokenizer_bert(
    review,
    max_length=64,   # tight budget to make the effect visible
    truncation=True,
    return_tensors='pt',
)

decoded = tokenizer_bert.decode(inputs_default['input_ids'][0], skip_special_tokens=True)
print('=== Tokens kept by DEFAULT (right) truncation ===')
print(decoded)
print()
print('Verdict present in kept tokens:', 'terrible' in decoded or 'throttles' in decoded)

**Explain Bug 2:** The training loss decreased because the model learned something from the kept tokens — but the label-bearing content was systematically cut. Why does right-truncation hurt accuracy specifically when the classification-relevant signal is at the end of documents? What types of tasks are safe to use with right-truncation and what types are not?

*(Write your answer here.)*

In [ ]:
# Fix Bug 2: keep the tail of the sequence (the end contains the verdict)
# Strategy: encode the full sequence, then keep the last max_length tokens

def tail_truncate(tokenizer, text, max_length):
    """Tokenise and keep the final max_length tokens (preserves end-of-document signal)."""
    all_ids = tokenizer.encode(text, add_special_tokens=False)
    # Keep only the last (max_length - 2) tokens to leave room for [CLS] and [SEP]
    keep = all_ids[-(max_length - 2):]
    # Re-add special tokens
    keep = [tokenizer.cls_token_id] + keep + [tokenizer.sep_token_id]
    return torch.tensor([keep])

input_ids_tail = tail_truncate(tokenizer_bert, review, max_length=64)
decoded_tail = tokenizer_bert.decode(input_ids_tail[0], skip_special_tokens=True)
print('=== Tokens kept by TAIL truncation ===')
print(decoded_tail)
print()
print('Verdict present in kept tokens:', 'terrible' in decoded_tail or 'throttles' in decoded_tail)

## Summary

> **For each bug, write one sentence on what went wrong and how to catch it early.**

1. 
2. 